# Lab 2 — The Lie Detector
**Session 2 · Prompt engineering + your first eval harness · TCE**

You'll need: your **10-question file from Lab 1 Part E**.
First: **File → Save a copy in Drive**.

In [ ]:
# Cell 1 — setup (same as Lab 1)
%pip install -q -U google-genai
from getpass import getpass
from google import genai
from google.genai import types
import time

client = genai.Client(api_key=getpass("Gemini API key: "))
MODEL = "gemini-flash-latest"  # the free tier's current Flash (July 2026 → Gemini 3.5 Flash). 503 'high demand'? swap to "gemini-flash-lite-latest".

def ask(prompt, temperature=None):
    config = types.GenerateContentConfig(temperature=temperature) if temperature is not None else None
    for attempt in range(4):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt, config=config).text
        except Exception as e:
            if "429" in str(e) and attempt < 3:
                print("rate limited, waiting..."); time.sleep(20*(attempt+1))
            else: raise
print("ready ✓")

## Part A — The prompt makeover (5 iterations)

Below is a deliberately terrible prompt. Improve it **five times**, one upgrade per run:
**v1** task (length+subject) → **v2** role+audience → **v3** context (real facts) → **v4** format+negative instructions → **v5** constraint ("only stated facts").

Run, read, then edit `PROMPT` and run again. Document each step in the table below.

In [ ]:
# Cell 2 — edit PROMPT, run, repeat (keep old versions in comments!)
PROMPT = "write about tce"   # v0 — terrible on purpose

print(ask(PROMPT))

### Document your makeover (edit this cell)

| v | What I changed | Why the output got better |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |

### ✓ Checkpoint 1 — five documented iterations.

---
## Part B — Your test set → your first eval

Fill `my_tests` from your Lab 1 Part E file. Keep `expected` SHORT — the key fact only (a name, a number), not a full sentence. The scorer checks whether your expected string appears inside the model's answer (after normalization).

In [ ]:
# Cell 3 — your 10 questions (2 examples shown — replace with YOURS)
my_tests = [
    {"q": "Who composed the music for the film Roja?", "expected": "rahman"},
    {"q": "In which year was TCE Madurai founded?",     "expected": "1957"},
    # ... add your 8+ more ...
]
print(len(my_tests), "questions loaded")

In [ ]:
# Cell 4 — the eval harness
import re

def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", s.lower())

def run_eval(template, tests, verbose=True):
    hits = 0
    for t in tests:
        ans = ask(template.format(q=t["q"]), temperature=0.0)
        ok = norm(t["expected"]) in norm(ans)
        hits += ok
        if verbose:
            print(("✓" if ok else "✗"), t["q"])
            if not ok:
                print("   expected:", t["expected"], "| got:", ans[:120].replace("\n"," "))
    score = hits / len(tests)
    print(f"\nSCORE: {hits}/{len(tests)} = {score:.0%}")
    return score

baseline = run_eval("Answer this question: {q}", my_tests)

### Read every ✗ before moving on
For each failure, decide: **model wrong** (hallucination — the interesting case), **scorer too strict** (fix your `expected` string), or **question ambiguous** (fix the question). This diagnosis IS the skill.

### ✓ Checkpoint 2 — eval ran on your 10 questions; failures diagnosed.

---
## Part C — Prompt A vs Prompt B, settled with numbers

In [ ]:
# Cell 5 — design a better template, then fight
PROMPT_A = "Answer this question: {q}"

PROMPT_B = (
    "You are a careful expert. Answer the question below.\n"
    "Rules: be direct, give the specific fact asked for, "
    "and if you are not sure, say 'I am not sure' instead of guessing.\n\n"
    "Question: {q}\nAnswer:"
)   # ← edit me — beat A by more!

print("=== A ==="); score_a = run_eval(PROMPT_A, my_tests, verbose=False)
print("=== B ==="); score_b = run_eval(PROMPT_B, my_tests, verbose=False)
print(f"\nA: {score_a:.0%}  vs  B: {score_b:.0%}  →  {'B wins' if score_b>score_a else 'A wins or tie — iterate B!'}")

### ✓ Checkpoint 3 — show me A vs B numbers + the single most interesting failure you found.

---
## Stretch goals

In [ ]:
# Stretch 1 — variance: is your score stable?
scores = [run_eval(PROMPT_B, my_tests, verbose=False) for _ in range(3)]
print("three runs:", [f"{s:.0%}" for s in scores], "| average:", f"{sum(scores)/3:.0%}")
# We used temperature=0.0 in the harness — try changing it in run_eval and watch stability change.

In [ ]:
# Stretch 2 — LLM-as-judge (a model grades a model)
def judge_score(question, expected, answer):
    verdict = ask(
        f"Question: {question}\nExpected key fact: {expected}\nStudent answer: {answer}\n"
        "Does the student answer contain the expected fact (paraphrase ok)? Reply only PASS or FAIL.",
        temperature=0.0)
    return "PASS" in verdict.upper()

t = my_tests[0]
ans = ask(PROMPT_B.format(q=t["q"]))
print("judge says:", judge_score(t["q"], t["expected"], ans))
# Now: where might the judge itself be wrong? (verbosity bias, self-agreement...)

## Wrap
You now own the loop: **prompt → eval → read failures → fix → re-run.** Keep this notebook — the same harness grades your capstone in Session 6.

**Short break. Session 3: AI gets eyes and ears — have a photo or two on your phone.**